# SynapseClusterEM in Google Colab

This notebook demonstrates how to run the SynapseClusterEM project in Google Colab.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alim98/SynapseClusterEM/blob/main/colab_setup.ipynb)

## 1. Clone the Repository

First, let's clone your GitHub repository:

In [ ]:
!git clone https://github.com/alim98/SynapseClusterEM.git
%cd SynapseClusterEM

## 2. Install Dependencies

Install the required packages:

In [ ]:
!pip install torch torchvision scikit-learn umap-learn plotly matplotlib pandas tqdm pillow imageio

In [ ]:
# Install the package in development mode
!pip install -e .

## 3. Mount Google Drive

Mount Google Drive to access your data and save results:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Set Up Data Paths

Define the paths to your data. You should update these paths to match your Google Drive structure.

In [ ]:
# Update these paths to match your Google Drive structure
RAW_BASE_DIR = "/content/drive/MyDrive/SynapseClusterEM/data/raw"
SEG_BASE_DIR = "/content/drive/MyDrive/SynapseClusterEM/data/seg"
ADD_MASK_BASE_DIR = "/content/drive/MyDrive/SynapseClusterEM/data/mask"
EXCEL_DIR = "/content/drive/MyDrive/SynapseClusterEM/data/excel"
CHECKPOINT_PATH = "/content/drive/MyDrive/SynapseClusterEM/models/vgg3d_checkpoint.pth"
OUTPUT_DIR = "/content/drive/MyDrive/SynapseClusterEM/outputs"

# Create output directories
!mkdir -p {OUTPUT_DIR}/global_norm
!mkdir -p {OUTPUT_DIR}/gif_visualization
!mkdir -p {OUTPUT_DIR}/analysis

## 5. Check Data Availability

Let's check if the data directories exist and list their contents:

In [ ]:
import os

def check_directory(path, name):
    if os.path.exists(path):
        print(f"✅ {name} directory exists at: {path}")
        print(f"Contents: {os.listdir(path)[:5]}" + ("..." if len(os.listdir(path)) > 5 else ""))
    else:
        print(f"❌ {name} directory not found at: {path}")
        # Create the directory
        os.makedirs(path, exist_ok=True)
        print(f"   Created empty directory at: {path}")
    print("")

print("Checking data directories...\n")
check_directory(RAW_BASE_DIR, "Raw data")
check_directory(SEG_BASE_DIR, "Segmentation data")
check_directory(ADD_MASK_BASE_DIR, "Additional mask data")
check_directory(EXCEL_DIR, "Excel files")

# Check if the model checkpoint exists
if os.path.exists(CHECKPOINT_PATH):
    print(f"✅ Model checkpoint exists at: {CHECKPOINT_PATH}")
else:
    print(f"❌ Model checkpoint not found at: {CHECKPOINT_PATH}")
    print("   You will need to upload your model checkpoint to this location.")

## 6. Upload Data (If Needed)

If your data is not already in Google Drive, you can upload it here:

In [ ]:
from google.colab import files

# Uncomment and run these cells if you need to upload data

# Upload raw data
# uploaded = files.upload()
# for filename in uploaded.keys():
#     !mv "{filename}" "{RAW_BASE_DIR}/"

# Upload segmentation data
# uploaded = files.upload()
# for filename in uploaded.keys():
#     !mv "{filename}" "{SEG_BASE_DIR}/"

# Upload additional mask data
# uploaded = files.upload()
# for filename in uploaded.keys():
#     !mv "{filename}" "{ADD_MASK_BASE_DIR}/"

# Upload Excel files
# uploaded = files.upload()
# for filename in uploaded.keys():
#     !mv "{filename}" "{EXCEL_DIR}/"

# Upload model checkpoint
# uploaded = files.upload()
# for filename in uploaded.keys():
#     !mv "{filename}" "{CHECKPOINT_PATH}"

## 7. Run Global Normalization

Calculate global normalization statistics:

In [ ]:
# Define the bounding box names and segmentation type
BBOX_NAMES = "bbox1 bbox2 bbox3 bbox4 bbox5 bbox6 bbox7"
SEGMENTATION_TYPE = 1  # Change this to your desired segmentation type

!python scripts/global_norm_example.py \
    --raw_base_dir "{RAW_BASE_DIR}" \
    --seg_base_dir "{SEG_BASE_DIR}" \
    --add_mask_base_dir "{ADD_MASK_BASE_DIR}" \
    --excel_dir "{EXCEL_DIR}" \
    --output_dir "{OUTPUT_DIR}/global_norm" \
    --segmentation_type {SEGMENTATION_TYPE}

## 8. Check Global Normalization Results

Let's check if the global statistics were calculated successfully:

In [ ]:
import json

GLOBAL_STATS_PATH = f"{OUTPUT_DIR}/global_norm/global_stats.json"

if os.path.exists(GLOBAL_STATS_PATH):
    with open(GLOBAL_STATS_PATH, 'r') as f:
        global_stats = json.load(f)
    print("Global statistics calculated successfully:")
    print(f"Mean: {global_stats['mean']}")
    print(f"Std: {global_stats['std']}")
else:
    print(f"Global statistics file not found at: {GLOBAL_STATS_PATH}")
    print("Please check the previous step for errors.")

## 9. Run Analysis with Global Normalization

Run the main analysis script with global normalization:

In [ ]:
# Define parameters for analysis
SEGMENTATION_TYPES = "1 2 3"  # Change these to your desired segmentation types
ALPHAS = "1.0"
N_CLUSTERS = 10
BATCH_SIZE = 4
NUM_WORKERS = 2

!python scripts/run_analysis.py \
    --raw_base_dir "{RAW_BASE_DIR}" \
    --seg_base_dir "{SEG_BASE_DIR}" \
    --add_mask_base_dir "{ADD_MASK_BASE_DIR}" \
    --excel_dir "{EXCEL_DIR}" \
    --checkpoint_path "{CHECKPOINT_PATH}" \
    --bbox_names {BBOX_NAMES} \
    --segmentation_types {SEGMENTATION_TYPES} \
    --alphas {ALPHAS} \
    --output_dir "{OUTPUT_DIR}/analysis" \
    --use_global_norm \
    --global_stats_path "{GLOBAL_STATS_PATH}" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --n_clusters {N_CLUSTERS}

## 10. Visualize Samples

Create GIF visualizations of specific samples:

In [ ]:
# Define parameters for visualization
BBOX_NAME = "bbox1"  # Change this to your desired bounding box
SAMPLE_INDEX = 0     # Change this to your desired sample index
SEG_TYPE = 1         # Change this to your desired segmentation type
ALPHA = 1.0
GRAY_VALUE = 0.5
FPS = 10

!python scripts/visualize_sample_as_gif.py \
    --raw_base_dir "{RAW_BASE_DIR}" \
    --seg_base_dir "{SEG_BASE_DIR}" \
    --add_mask_base_dir "{ADD_MASK_BASE_DIR}" \
    --excel_dir "{EXCEL_DIR}" \
    --bbox_name {BBOX_NAME} \
    --sample_index {SAMPLE_INDEX} \
    --segmentation_type {SEG_TYPE} \
    --alpha {ALPHA} \
    --gray_value {GRAY_VALUE} \
    --fps {FPS} \
    --output_dir "{OUTPUT_DIR}/gif_visualization"

## 11. Display Results

Display the generated visualizations:

In [ ]:
from IPython.display import Image, display, HTML
import glob

# Display a GIF visualization
gif_path = f"{OUTPUT_DIR}/gif_visualization/sample_{SAMPLE_INDEX}_seg{SEG_TYPE}_alpha{ALPHA}_gray{GRAY_VALUE}.gif"
if os.path.exists(gif_path):
    display(HTML(f'<h3>GIF Visualization of Sample {SAMPLE_INDEX} with Segmentation Type {SEG_TYPE}</h3>'))
    display(Image(gif_path))
else:
    print(f"GIF not found at {gif_path}")
    # Try to find any GIFs in the directory
    gif_files = glob.glob(f"{OUTPUT_DIR}/gif_visualization/*.gif")
    if gif_files:
        print(f"Found other GIFs: {gif_files}")
        display(HTML(f'<h3>Available GIF Visualization</h3>'))
        display(Image(gif_files[0]))
    else:
        print("No GIFs found in the output directory.")

## 12. Load and Explore Clustering Results

Load and visualize the clustering results:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

# Find the segmentation type result directories
seg_dirs = glob.glob(f"{OUTPUT_DIR}/analysis/seg*")
if seg_dirs:
    # Display results for each segmentation type
    for seg_dir in seg_dirs:
        seg_type = os.path.basename(seg_dir).replace('seg', '')
        
        # Load clustering results
        cluster_csv = f"{seg_dir}/features_with_clusters.csv"
        if os.path.exists(cluster_csv):
            df = pd.read_csv(cluster_csv)
            display(HTML(f'<h3>Clustering Results for Segmentation Type {seg_type}</h3>'))
            print(f"Loaded clustering results with {len(df)} samples")
            
            # Display the first few rows
            display(df.head())
            
            # Plot UMAP projection with cluster colors
            if 'umap_1' in df.columns and 'umap_2' in df.columns and 'cluster' in df.columns:
                fig = px.scatter(df, x='umap_1', y='umap_2', color='cluster', 
                               hover_data=['sample_index', 'bbox_name'],
                               title=f'UMAP Projection with Cluster Assignments (Segmentation Type {seg_type})')
                fig.show()
                
                # Plot cluster distribution
                plt.figure(figsize=(10, 6))
                cluster_counts = df['cluster'].value_counts().sort_index()
                plt.bar(cluster_counts.index, cluster_counts.values)
                plt.xlabel('Cluster')
                plt.ylabel('Number of Samples')
                plt.title(f'Cluster Distribution (Segmentation Type {seg_type})')
                plt.xticks(range(N_CLUSTERS))
                plt.grid(axis='y', linestyle='--', alpha=0.7)
                plt.show()
        else:
            print(f"Cluster CSV file not found: {cluster_csv}")
else:
    print(f"No segmentation results found in {OUTPUT_DIR}/analysis")

## 13. Visualize Representative Samples from Each Cluster

Create visualizations for representative samples from each cluster:

In [ ]:
# Choose a segmentation type to visualize
SEG_TYPE_TO_VISUALIZE = 1  # Change this to your desired segmentation type

# Find the segmentation type result directory
seg_dir = f"{OUTPUT_DIR}/analysis/seg{SEG_TYPE_TO_VISUALIZE}"
if os.path.exists(seg_dir):
    # Load clustering results
    cluster_csv = f"{seg_dir}/features_with_clusters.csv"
    if os.path.exists(cluster_csv):
        df = pd.read_csv(cluster_csv)
        
        # For each cluster, visualize a representative sample
        for cluster in range(N_CLUSTERS):
            # Get samples from this cluster
            cluster_samples = df[df['cluster'] == cluster]
            
            if len(cluster_samples) > 0:
                # Get a representative sample (first one for simplicity)
                sample = cluster_samples.iloc[0]
                bbox_name = sample['bbox_name']
                sample_index = int(sample['sample_index'])
                
                print(f"Visualizing representative sample from cluster {cluster} (bbox: {bbox_name}, index: {sample_index})")
                
                # Create output directory for this cluster
                cluster_output_dir = f"{OUTPUT_DIR}/gif_visualization/cluster_{cluster}"
                !mkdir -p {cluster_output_dir}
                
                # Run visualization for this sample
                !python scripts/visualize_sample_as_gif.py \
                    --raw_base_dir "{RAW_BASE_DIR}" \
                    --seg_base_dir "{SEG_BASE_DIR}" \
                    --add_mask_base_dir "{ADD_MASK_BASE_DIR}" \
                    --excel_dir "{EXCEL_DIR}" \
                    --bbox_name "{bbox_name}" \
                    --sample_index {sample_index} \
                    --segmentation_type {SEG_TYPE_TO_VISUALIZE} \
                    --alpha 1.0 \
                    --gray_value 0.5 \
                    --fps 10 \
                    --output_dir "{cluster_output_dir}"
    else:
        print(f"Cluster CSV file not found: {cluster_csv}")
else:
    print(f"No results found for segmentation type {SEG_TYPE_TO_VISUALIZE} in {OUTPUT_DIR}/analysis")

## 14. Display Representative Samples from Each Cluster

Display the GIF visualizations of representative samples from each cluster:

In [ ]:
# Display GIFs for each cluster
for cluster in range(N_CLUSTERS):
    cluster_output_dir = f"{OUTPUT_DIR}/gif_visualization/cluster_{cluster}"
    gif_files = glob.glob(f"{cluster_output_dir}/*.gif")
    
    if gif_files:
        display(HTML(f'<h3>Representative Sample from Cluster {cluster}</h3>'))
        display(Image(gif_files[0]))
    else:
        print(f"No GIFs found for cluster {cluster}")

## 15. Save Results Back to Google Drive

If you've made any changes or generated new results that you want to save back to Google Drive:

In [ ]:
# Zip the results for easy download
!zip -r /content/synapse_results.zip {OUTPUT_DIR}

# Download the zipped results
from google.colab import files
files.download('/content/synapse_results.zip')

## 16. Run the Complete Workflow Script (Optional)

Alternatively, you can run the complete workflow script that performs all the steps in sequence:

In [ ]:
# First, update the workflow script with the correct paths
!sed -i "s|RAW_BASE_DIR=\".*\"|RAW_BASE_DIR=\"{RAW_BASE_DIR}\"|g" scripts/workflow.sh
!sed -i "s|SEG_BASE_DIR=\".*\"|SEG_BASE_DIR=\"{SEG_BASE_DIR}\"|g" scripts/workflow.sh
!sed -i "s|ADD_MASK_BASE_DIR=\".*\"|ADD_MASK_BASE_DIR=\"{ADD_MASK_BASE_DIR}\"|g" scripts/workflow.sh
!sed -i "s|EXCEL_DIR=\".*\"|EXCEL_DIR=\"{EXCEL_DIR}\"|g" scripts/workflow.sh
!sed -i "s|CHECKPOINT_PATH=\".*\"|CHECKPOINT_PATH=\"{CHECKPOINT_PATH}\"|g" scripts/workflow.sh
!sed -i "s|OUTPUT_DIR=\".*\"|OUTPUT_DIR=\"{OUTPUT_DIR}/workflow_results\"|g" scripts/workflow.sh

# Make the script executable
!chmod +x scripts/workflow.sh

# Run the workflow script
# Uncomment the line below to run the complete workflow
# !./scripts/workflow.sh

## Conclusion

This notebook has demonstrated how to run the SynapseClusterEM project in Google Colab. You've learned how to:

1. Set up the environment and install dependencies
2. Access data from Google Drive
3. Calculate global normalization statistics
4. Run the main analysis with global normalization
5. Visualize samples and clustering results
6. Save results back to Google Drive

You can now use this notebook as a template for your own analysis, modifying the parameters and paths as needed.